In [1]:
import os
import re
import spacy
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

C:\Users\HP\AppData\Local\Temp\ipykernel_25136\888630855.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


In [2]:
loader = PyPDFLoader('nitish_resume_A.pdf')
text = loader.load()
pages = []
for page in text:
    pages.append(page.page_content)
pages
data = ""
for i in pages:
    data += i
data = data.lower()

In [3]:
nlp = spacy.load('en_core_web_sm')

In [4]:
tokens = nlp(data)
lemmatized_tokens = [token.lemma_ for token in tokens if not token.is_stop]
data = " ".join(lemmatized_tokens).strip()
data

'nitish kumar gupta \n phone +91 9576273910 | envelope kumarknitish@gmail.com | linkedin linkedin.com/in/nitish-kumar-gupta-1b4274269 \n | github github.com/nitishkrgupta | globe nitishkrgupta.github.io/portfolio \n profile summary \n ai engineer strong foundation artiﬁcial intelligence , machine learning , deep learning . \n experience develop end - - end ai solution nlp computer vision . proﬁcient model \n development , optimization , deployment modern ai framework . \n internship \n codenscious.ai jul 2024 – oct 2024 \n computer vision intern indore , india \n – perform datum preprocessing , cleaning , annotation , augmentation improve model performance . \n – develop optimize deep learning - base computer vision model , improve accuracy \n hyperparameter tuning . \n – collaborate team member improve model eﬃciency ensure smooth deployment readiness . \n project \n intelligent question generator ( nlp ) \n – develop nlp - base question generation system streamlit llama generate mcqs

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100,
    chunk_overlap = 10
)
data = text_splitter.create_documents([data])
# data

In [6]:
embedding_model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-miniLM-L6-V2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
vectordb = FAISS.from_documents(documents=data, embedding=embedding_model)
vectordb

In [8]:
job_description = TextLoader('job_description.txt')
text = job_description.load()
job = []
for page in text:
    job.append(page.page_content)
data = ""
for job_data in job:
    data += job_data

job_description = data.lower()
job_description
print(type(job_description))

<class 'str'>


In [9]:
ats_prompt = ChatPromptTemplate.from_messages([
    ("system",
        """
You are an expert ATS (Applicant Tracking System) resume evaluator and document verifier.

Your task is to evaluate how well the RETRIEVED DOCUMENT matches the provided JOB DESCRIPTION and calculate an ATS match score from 0 to 100.

IMPORTANT RULES:
1. Evaluate ONLY the information explicitly present in the retrieved document.
2. Do NOT assume, infer, or invent skills, qualifications, experience, or achievements that are not present.
3. Compare the candidate's document against the job description across:
   - Required technical skills
   - Preferred technical skills
   - Relevant work experience
   - Years of experience
   - Education and certifications
   - Domain/industry experience
   - Tools, frameworks, programming languages, and technologies
   - Responsibilities and role alignment
   - Relevant projects
4. Give higher weight to REQUIRED qualifications and core skills than preferred qualifications.
5. Exact or strong semantic matches should receive credit. For example, "computer vision" and "OpenCV-based image processing" may be related, but do not treat unrelated technologies as equivalent.
6. Missing required skills should negatively affect the score.
7. Do not penalize the candidate for information that is not required by the job description.
8. Ignore irrelevant information in the retrieved document.
9. The score must represent the overall match between the document and the job description, not the quality of the resume/document itself.
10. Keep the score realistic. Do not give a high score simply because some keywords match.
11. If the retrieved document contains insufficient information to evaluate a requirement, treat that requirement as NOT CONFIRMED rather than assuming the candidate has it.
12. Return only the requested output format. Do not add introductory or concluding remarks.

SCORING GUIDELINE:
- 90–100: Excellent match; candidate clearly satisfies almost all important requirements.
- 80–89: Strong match; candidate satisfies most critical requirements with only minor gaps.
- 70–79: Moderate match; several relevant requirements are met, but there are noticeable gaps.
- 50–69: Weak match; some relevant skills are present, but multiple important requirements are missing.
- 0–49: Poor match; limited alignment with the job description.

SCORING PRIORITY:
- Required skills/qualifications: highest weight
- Relevant experience/responsibilities: high weight
- Core tools/technologies: high weight
- Education/certifications: medium weight
- Preferred/nice-to-have skills: lower weight

IF SCORE >= 80:
Return EXACTLY this structure:

Score - <score>%
Highlight Key Skills -
- <skill/requirement matched>
- <skill/requirement matched>
- <skill/requirement matched>

Relevant Chunk -
<Quote or reproduce only the most relevant portions of the retrieved document that support the score. Do not invent text.>

Summarise the relevant chunk -
<Concise summary explaining why these sections are relevant to the job description.>

IF SCORE < 80:
Return EXACTLY this structure:

Score - <score>%

Reason for low score -
<Clearly explain the major gaps between the retrieved document and the job description. Mention missing or insufficiently demonstrated required skills, experience, qualifications, or technologies.>

Suggestions to improve -
- <Specific improvement based on a missing job requirement>
- <Specific improvement based on a missing or weakly demonstrated skill>
- <Specific improvement based on experience/project alignment>
- <Specific improvement based on keywords or technologies missing from the document>

Do not recommend adding a skill unless the candidate genuinely has that skill or can acquire it legitimately. Never suggest falsely claiming experience.
"""),
    ("human", """JOB DESCRIPTION : {job_description}, RETRIEVED DOCUMENT : {r_chunks}

Evaluate the retrieved document against the job description using the scoring rules above.
"""
    )
])

In [10]:
llm_model = ChatGoogleGenerativeAI(
    model = 'gemini-3.5-flash-lite',
    api_key = os.environ['GEMINI_API_KEY']
)

In [11]:
parser = StrOutputParser()

In [12]:
ats_chain = ats_prompt | llm_model | parser

In [13]:
def resume_analyzer(job_description):
    r_chunks = vectordb.similarity_search(job_description)
    r_chunks = [doc.page_content for doc in r_chunks]

    response = ats_chain.invoke({'job_description' : job_description, 'r_chunks' : r_chunks})

    return response

res = resume_analyzer(job_description)
print(res)

Score - 68%

Reason for low score -
- The retrieved document completely lacks mention of core required skills such as Python programming, pandas, NumPy, data preprocessing, feature engineering, exploratory data analysis (EDA), and SQL knowledge.
- Essential machine learning concepts and tasks required by the job description (such as regression, classification, clustering, evaluation metrics, etc.) are not explicitly demonstrated in the text, outside of a general mention of machine learning and deep learning.
- There is no mention of handling structured/unstructured datasets, deployment of models as APIs, or documentation of experiments.

Suggestions to improve -
- Include explicit mentions of Python programming along with core data handling libraries like Pandas and NumPy.
- Detail specific machine learning projects demonstrating classification, regression, clustering, and model evaluation metrics (accuracy, precision, recall, RMSE, etc.).
- Add information regarding data preprocessing